# 02 — Exploratory Data Analysis (EDA)
## Peramalan Nilai PM2.5 Harian DKI Jakarta dengan SVR

Notebook ini murni untuk memahami karakteristik data, **tidak ada pemodelan di sini**. Input: `data/pm25_daily.csv` (hasil `01_data_wrangling.ipynb`).

Langkah-langkah:
1. Muat data dan cek ulang bentuknya
2. Statistik deskriptif dan distribusi `pm25`
3. Plot deret waktu penuh
4. Pola musiman (rata-rata per bulan, per hari dalam minggu)
5. Tren tahunan
6. Dekomposisi STL (tren, musiman, residual)
7. Uji stasioneritas (Augmented Dickey-Fuller)
8. ACF dan PACF — dasar pemilihan lag untuk notebook 03
9. Korelasi `pm25` dengan polutan lain
10. Ringkasan temuan

> **Catatan satuan:** nilai `pm25` adalah angka indeks ISPU, bukan konsentrasi µg/m³ (lihat `docs/dataset.md` bagian 3).

## 1. Setup dan Muat Data

In [ ]:
import os

if 'google.colab' in str(get_ipython()) and not os.path.exists('data'):
    # Ganti USERNAME dengan username GitHub Anda
    !git clone https://github.com/USERNAME/jakarta-pm25-svr-forecasting.git
    %cd jakarta-pm25-svr-forecasting

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 150)

IN_PATH = 'data/pm25_daily.csv'

In [ ]:
df = pd.read_csv(IN_PATH, parse_dates=['tanggal']).set_index('tanggal')
df = df.asfreq('D')  # pastikan frekuensi harian eksplisit untuk STL/ACF

print(f"Jumlah baris : {len(df):,}")
print(f"Rentang      : {df.index.min().date()} s.d. {df.index.max().date()}")
print(f"Missing pm25 : {df['pm25'].isna().sum()}")
df.head()

## 2. Statistik Deskriptif dan Distribusi

Melihat rentang nilai, kecenderungan pusat, dan apakah distribusinya miring (skewed) atau ada outlier.

In [ ]:
print(df['pm25'].describe().round(2))
print(f"\nSkewness: {df['pm25'].skew():.2f}")
print(f"Kurtosis: {df['pm25'].kurt():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['pm25'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribusi Nilai PM2.5')
axes[0].set_xlabel('Nilai ISPU PM2.5')
axes[0].set_ylabel('Frekuensi (hari)')

axes[1].boxplot(df['pm25'].dropna(), vert=True)
axes[1].set_title('Boxplot Nilai PM2.5')
axes[1].set_ylabel('Nilai ISPU PM2.5')

plt.tight_layout()
plt.savefig('docs/pm25_distribusi.png', dpi=120)
plt.show()

## 3. Plot Deret Waktu Penuh

Melihat pola keseluruhan: tren, lonjakan, dan periode data hilang (celah panjang pada notebook 01 akan terlihat sebagai bagian kosong pada grafik `pm10`, dsb., tapi `pm25` sendiri sudah bersih).

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df['pm25'], linewidth=0.7, color='steelblue')
ax.set_title('Nilai PM2.5 Harian DKI Jakarta, 2021-2025')
ax.set_ylabel('Nilai ISPU PM2.5')
ax.set_xlabel('Tanggal')
plt.tight_layout()
plt.savefig('docs/pm25_trend.png', dpi=120)
plt.show()

## 4. Pola Musiman

### 4.1 Rata-rata per bulan

In [ ]:
rata_bulanan = df.groupby(df.index.month)['pm25'].agg(['mean', 'std']).round(1)
rata_bulanan.index.name = 'bulan'
print(rata_bulanan)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(rata_bulanan.index, rata_bulanan['mean'], yerr=rata_bulanan['std'],
       color='steelblue', capsize=3)
ax.set_xticks(range(1, 13))
ax.set_xlabel('Bulan')
ax.set_ylabel('Rata-rata Nilai PM2.5')
ax.set_title('Pola Musiman: Rata-rata PM2.5 per Bulan (± std)')
plt.tight_layout()
plt.savefig('docs/pm25_pola_bulanan.png', dpi=120)
plt.show()

### 4.2 Rata-rata per hari dalam minggu

In [ ]:
nama_hari = ['Senin', 'Selasa', 'Rabu', 'Kamis', "Jumat", 'Sabtu', 'Minggu']
rata_harian = df.groupby(df.index.dayofweek)['pm25'].mean().round(1)
rata_harian.index = nama_hari

print(rata_harian)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(rata_harian.index, rata_harian.values, color='darkorange')
ax.set_ylabel('Rata-rata Nilai PM2.5')
ax.set_title('Rata-rata PM2.5 per Hari dalam Minggu')
plt.tight_layout()
plt.show()

## 5. Tren Tahunan

Melihat apakah kualitas udara membaik atau memburuk dari tahun ke tahun. Perhatikan bahwa 2021 dan 2025 tidak mencakup satu tahun penuh.

In [ ]:
rata_tahunan = df.groupby(df.index.year)['pm25'].agg(['mean', 'std', 'count']).round(1)
rata_tahunan.index.name = 'tahun'
print(rata_tahunan)
print("\nCatatan: 2021 mulai dari Januari (lengkap), 2025 hanya sampai Februari (tidak penuh).")

## 6. Dekomposisi STL

STL (Seasonal-Trend decomposition using LOESS) memisahkan deret menjadi tiga komponen: tren, musiman, dan residual. Berguna untuk memastikan pola musiman yang terlihat di bagian 4 bukan kebetulan.

Missing value pada `pm25` sudah 0 (dari notebook 01), sehingga STL bisa langsung dipakai tanpa imputasi tambahan.

In [ ]:
stl = STL(df['pm25'], period=365, robust=True)
hasil_stl = stl.fit()

fig = hasil_stl.plot()
fig.set_size_inches(13, 8)
plt.tight_layout()
plt.savefig('docs/pm25_stl_decomposition.png', dpi=120)
plt.show()

## 7. Uji Stasioneritas (Augmented Dickey-Fuller)

Deret dikatakan stasioner jika rata-rata dan variansnya relatif stabil sepanjang waktu. Hipotesis nol (H0) ADF test: deret **tidak** stasioner (mengandung unit root). Jika p-value < 0.05, H0 ditolak → deret dianggap stasioner.

In [ ]:
hasil_adf = adfuller(df['pm25'].dropna())

print(f"ADF Statistic : {hasil_adf[0]:.4f}")
print(f"p-value       : {hasil_adf[1]:.4f}")
print(f"Lags digunakan: {hasil_adf[2]}")
print(f"Jumlah observasi: {hasil_adf[3]}")
print("Critical values:")
for key, val in hasil_adf[4].items():
    print(f"  {key}: {val:.4f}")

if hasil_adf[1] < 0.05:
    print("\nKesimpulan: p-value < 0.05, H0 ditolak → deret PM2.5 dianggap STASIONER.")
else:
    print("\nKesimpulan: p-value >= 0.05, H0 tidak ditolak → deret PM2.5 belum tentu stasioner.")

## 8. ACF dan PACF: Mengidentifikasi Kandidat Lag

**Penting:** ACF dan PACF di sini **tidak menetapkan** lag mana yang wajib dipakai. Keduanya hanya alat eksplorasi untuk melihat pola ketergantungan temporal pada data. Kandidat lag yang muncul dari analisis ini akan diuji ulang lewat performa model SVR di `04_svr_modeling.ipynb` — jika sebuah lag ternyata tidak membantu performa model, ia tidak perlu dipaksakan meskipun tampak signifikan secara statistik di sini.

- **ACF** (Autocorrelation Function): mengukur korelasi PM2.5 hari ini dengan *k* hari sebelumnya, termasuk pengaruh tidak langsung yang "menular" lewat lag-lag di antaranya.
- **PACF** (Partial ACF): mengukur korelasi langsung pada lag *k*, setelah pengaruh lag 1 sampai *k*-1 dihilangkan. Ini yang lebih relevan untuk memilih fitur lag, karena menghindari duplikasi informasi.

In [ ]:
from statsmodels.tsa.stattools import acf, pacf

NLAGS = 30

acf_values = acf(df['pm25'].dropna(), nlags=NLAGS, fft=True)
pacf_values = pacf(df['pm25'].dropna(), nlags=NLAGS, method='ywm')

n_obs = df['pm25'].dropna().shape[0]
batas_signifikan = 1.96 / np.sqrt(n_obs)

print(f"Jumlah observasi : {n_obs}")
print(f"Batas signifikansi (95%): ±{batas_signifikan:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_acf(df['pm25'].dropna(), lags=NLAGS, ax=axes[0])
axes[0].set_title('ACF — Nilai PM2.5')

plot_pacf(df['pm25'].dropna(), lags=NLAGS, ax=axes[1], method='ywm')
axes[1].set_title('PACF — Nilai PM2.5')

plt.tight_layout()
plt.savefig('docs/pm25_acf_pacf.png', dpi=120)
plt.show()

### Tabel signifikansi ACF/PACF per lag

Lag dianggap signifikan jika nilainya berada di luar pita ±batas signifikansi (garis biru pada grafik di atas).

In [ ]:
tabel_acf_pacf = pd.DataFrame({
    'lag': range(1, NLAGS + 1),
    'acf': acf_values[1:],
    'pacf': pacf_values[1:],
})
tabel_acf_pacf['acf_signifikan'] = tabel_acf_pacf['acf'].abs() > batas_signifikan
tabel_acf_pacf['pacf_signifikan'] = tabel_acf_pacf['pacf'].abs() > batas_signifikan

print(tabel_acf_pacf.round(3).to_string(index=False))

### Menentukan kandidat lag dari hasil aktual

**Catatan penting soal "signifikan secara statistik":** dengan jumlah data yang besar (n ≈ 1.500), batas signifikansi 95% menjadi sangat longgar (±0,05). Akibatnya, hampir semua lag ACF akan lolos uji signifikansi meski nilainya kecil — ini **bukan bukti kuat** bahwa lag tersebut penting secara praktis. Karena itu, tabel di bawah menampilkan **besaran (magnitude)** PACF secara eksplisit, bukan cuma status Ya/Tidak, supaya kandidat lag dipilih berdasarkan seberapa besar pengaruhnya, bukan sekadar lolos ambang statistik.

In [ ]:
# Urutkan berdasarkan besaran PACF (nilai absolut), bukan cuma status signifikan
tabel_pacf_terurut = (
    tabel_acf_pacf[['lag', 'pacf']]
    .assign(pacf_abs=lambda d: d['pacf'].abs())
    .sort_values('pacf_abs', ascending=False)
    .reset_index(drop=True)
)

print("Lag diurutkan dari pengaruh langsung (PACF) TERBESAR ke TERKECIL:\n")
print(tabel_pacf_terurut.head(15).round(3).to_string(index=False))

In [ ]:
# Kandidat lag diambil dari PACF signifikan (lolos ambang statistik),
# TAPI besarannya tetap dilaporkan terpisah supaya tidak disalahartikan
# sebagai "semuanya sama penting".
lag_acf_signifikan = tabel_acf_pacf.loc[tabel_acf_pacf['acf_signifikan'], 'lag'].tolist()
lag_pacf_signifikan = tabel_acf_pacf.loc[tabel_acf_pacf['pacf_signifikan'], 'lag'].tolist()

print(f"Lag dengan ACF signifikan : {lag_acf_signifikan}")
print(f"Lag dengan PACF signifikan: {lag_pacf_signifikan}")

# Kandidat LUAS untuk notebook 03 — sengaja tidak dipersempit di sini.
# Pemilihan fitur FINAL dilakukan lewat forward selection + TimeSeriesSplit
# di notebook 04, berdasarkan performa model, bukan ambang statistik ini.
lag_kandidat = lag_pacf_signifikan
print(f"\n>>> Kandidat lag (luas, belum final) untuk notebook 03: {lag_kandidat}")

**Interpretasi (diperbaiki):** Dari tabel besaran di atas, lag 1–4 memiliki PACF yang jauh lebih besar (0,66 turun ke sekitar 0,11) dibanding lag lain. Lag-lag yang lebih jauh (mis. 7, 14, 21) juga lolos ambang signifikansi, tetapi besarannya jauh lebih kecil (sekitar 0,05–0,09) — **lolos uji statistik karena jumlah data besar, bukan karena pengaruhnya besar secara praktis**. Karena itu, daftar `lag_kandidat` di atas sengaja dibuat **luas** (bukan dipersempit manual di sini) dan akan disaring lebih lanjut lewat **forward selection dengan `TimeSeriesSplit`** di `04_svr_modeling.ipynb`, berdasarkan kontribusi nyata tiap lag terhadap penurunan RMSE — bukan berdasarkan ambang ACF/PACF semata.

## 9. Korelasi dengan Polutan Lain

Menentukan variabel mana yang layak diuji sebagai eksperimen tambahan di `04_svr_modeling.ipynb`.

In [ ]:
korelasi = df.corr()
print(korelasi['pm25'].round(2).sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(korelasi, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(korelasi.columns)))
ax.set_yticks(range(len(korelasi.columns)))
ax.set_xticklabels(korelasi.columns, rotation=45, ha='right')
ax.set_yticklabels(korelasi.columns)
for i in range(len(korelasi)):
    for j in range(len(korelasi)):
        ax.text(j, i, f"{korelasi.iloc[i, j]:.2f}", ha='center', va='center', fontsize=8)
ax.set_title('Korelasi Antar Polutan')
plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
plt.savefig('docs/korelasi_polutan.png', dpi=120)
plt.show()

## 10. Ringkasan Temuan

Sel ini mencetak ringkasan otomatis dari semua analisis di atas, untuk disalin ke README atau laporan.

In [ ]:
print("=" * 60)
print("RINGKASAN TEMUAN EDA")
print("=" * 60)
print(f"Rentang data       : {df.index.min().date()} s.d. {df.index.max().date()}")
print(f"Rata-rata pm25     : {df['pm25'].mean():.1f}")
print(f"Std pm25           : {df['pm25'].std():.1f}")
print(f"Skewness           : {df['pm25'].skew():.2f}")
print(f"Bulan tertinggi    : {rata_bulanan['mean'].idxmax()} (rata-rata {rata_bulanan['mean'].max():.1f})")
print(f"Bulan terendah     : {rata_bulanan['mean'].idxmin()} (rata-rata {rata_bulanan['mean'].min():.1f})")
print(f"Tahun tertinggi    : {rata_tahunan['mean'].idxmax()} (rata-rata {rata_tahunan['mean'].max():.1f})")
print(f"Tahun terendah     : {rata_tahunan['mean'].idxmin()} (rata-rata {rata_tahunan['mean'].min():.1f})")
print(f"ADF p-value        : {hasil_adf[1]:.4f} ({'stasioner' if hasil_adf[1] < 0.05 else 'belum tentu stasioner'})")
print(f"Kandidat lag (dari PACF): {lag_kandidat}")
print(f"Korelasi dgn pm10  : {korelasi.loc['pm10', 'pm25']:.2f}")
print(f"Korelasi dgn co    : {korelasi.loc['co', 'pm25']:.2f}")
print(f"Korelasi dgn o3    : {korelasi.loc['o3', 'pm25']:.2f}")
print("=" * 60)

## Ringkasan

| Aspek | Temuan (lihat output Sel 10 untuk angka pasti) |
|---|---|
| Distribusi | Dicetak otomatis (skewness, rentang) |
| Pola musiman | Grafik `docs/pm25_pola_bulanan.png` |
| Tren tahunan | Tabel pada Sel 12 |
| Stasioneritas | Hasil ADF test pada Sel 15 |
| Lag yang relevan | Ditentukan dari ACF/PACF pada Sel 17-18 |
| Polutan paling berkorelasi | `docs/korelasi_polutan.png` |

**Tidak ada file CSV baru dari notebook ini** — hanya analisis dan gambar. Notebook `03_feature_engineering.ipynb` tetap membaca `data/pm25_daily.csv` yang sama.

**Langkah berikutnya:** buka `03_feature_engineering.ipynb` untuk membentuk fitur lag/rolling dan membagi data latih/uji.